In [1]:
"""
compare_transitions.py

Compares the raw transition-rate data exported from the Python MPRE
simulation (raw_transition_data.npz) against the corresponding PICWave-style
CSV exports (*.txt) for each transition (WL <-> ES2 <-> ES1 <-> GS).

Usage:
    Place all of the following files inside a folder named "comparisons"
    (next to this script) and run:

        python compare_transitions.py

    Files expected in comparisons/:
        raw_transition_data.npz
        WL_ES2.txt, ES2_WL.txt, ES2_ES1.txt, ES1_ES2.txt, ES1_GS.txt, GS_ES1.txt

Output:
    For each transition: printed quantitative comparison stats (computed on
    the overlapping x-range only, since the two sources use different
    sampling grids and extents), plus an overlay plot saved as
    comparisons/compare_<TRANSITION>.png

    Note: y-values from both sources are taken as |y| before comparison, so
    sign-convention differences between the two data sources don't affect
    the reported diffs.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

NPZ_FILE = "raw_transition_data.npz"

# Maps each txt filename ("From_To") to the matching npz key suffix.
# (WL is abbreviated to "w" in the npz key names.)
TRANSITION_MAP = {
    "WL_ES2":  "w_es2",
    "ES2_WL":  "es2_w",
    "ES2_ES1": "es2_es1",
    "ES1_ES2": "es1_es2",
    "ES1_GS":  "es1_gs",
    "GS_ES1":  "gs_es1",
}


def load_txt(path):
    """Load a 2-column 'Xvals, Yvals1,' CSV, scale x by 1E18 to match the npz units,
    and take |y| so sign-convention differences don't affect the comparison."""
    data = np.genfromtxt(path, delimiter=",", skip_header=1, usecols=(0, 1))
    x = data[:, 0] * 1e18
    y = np.abs(data[:, 1])
    mask = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[mask], y[mask]
    order = np.argsort(x)
    return x[order], y[order]


def compare_transition(name_txt, name_npz, npz_data):
    x_txt, y_txt = load_txt(os.path.join(f"{name_txt}.txt"))

    x_npz = npz_data[f"x_{name_npz}"]
    y_npz = np.abs(npz_data[f"y_{name_npz}"])
    order = np.argsort(x_npz)
    x_npz, y_npz = x_npz[order], y_npz[order]

    # Only compare over the x-range the two datasets actually share.
    lo, hi = max(x_txt.min(), x_npz.min()), min(x_txt.max(), x_npz.max())

    stats = None
    if hi <= lo:
        print(f"[{name_txt}] No overlapping x-range between txt and npz data - skipping stats.")
    else:
        x_common = np.linspace(lo, hi, 500)
        y_txt_i = np.interp(x_common, x_txt, y_txt)
        y_npz_i = np.interp(x_common, x_npz, y_npz)

        abs_diff = np.abs(y_txt_i - y_npz_i)
        denom = np.maximum(y_npz_i, 1e-30)
        rel_diff = abs_diff / denom

        stats = {
            "overlap_lo": lo,
            "overlap_hi": hi,
            "max_abs_diff": abs_diff.max(),
            "mean_abs_diff": abs_diff.mean(),
            "max_rel_diff_pct": rel_diff.max() * 100,
            "mean_rel_diff_pct": rel_diff.mean() * 100,
        }

        print(f"[{name_txt}]  overlap x-range: {lo:.3e} to {hi:.3e}")
        print(f"    max |diff|    = {stats['max_abs_diff']:.4e}")
        print(f"    mean |diff|   = {stats['mean_abs_diff']:.4e}")
        print(f"    max rel diff  = {stats['max_rel_diff_pct']:.2f} %")
        print(f"    mean rel diff = {stats['mean_rel_diff_pct']:.2f} %")

    # Overlay plot (always produced, even without a numeric overlap)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(x_npz, y_npz, 'o', ms=3, alpha=0.6, label=f"npz: {name_npz}")
    ax.plot(x_txt, y_txt, '-', lw=1.5, label=f"txt: {name_txt}")
    ax.set_xlabel("Carrier density (m$^{-3}$)")
    ax.set_ylabel("|Rate|")
    ax.set_title(f"Transition comparison: {name_txt}")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    out_path = os.path.join(f"compare_{name_txt}.png")
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"    Saved plot -> {out_path}\n")

    return stats


def main():
    npz_data = np.load(os.path.join(NPZ_FILE))
    results = {}
    for name_txt, name_npz in TRANSITION_MAP.items():
        results[name_txt] = compare_transition(name_txt, name_npz, npz_data)
    return results


if __name__ == "__main__":
    main()

[WL_ES2]  overlap x-range: 2.507e+14 to 6.000e+16
    max |diff|    = 7.1399e+25
    mean |diff|   = 9.4519e+24
    max rel diff  = 0.92 %
    mean rel diff = 0.12 %
    Saved plot -> compare_WL_ES2.png

[ES2_WL]  overlap x-range: 4.640e+15 to 7.068e+17
    max |diff|    = 4.4922e+22
    mean |diff|   = 7.3311e+21
    max rel diff  = 0.00 %
    mean rel diff = 0.00 %
    Saved plot -> compare_ES2_WL.png

[ES2_ES1]  overlap x-range: 4.640e+15 to 7.068e+17
    max |diff|    = 1.6976e+27
    mean |diff|   = 1.9544e+26
    max rel diff  = 70.63 %
    mean rel diff = 1.23 %
    Saved plot -> compare_ES2_ES1.png

[ES1_ES2]  overlap x-range: 1.640e+16 to 4.688e+17
    max |diff|    = 4.4874e+26
    mean |diff|   = 5.3426e+25
    max rel diff  = 424.01 %
    mean rel diff = 1.20 %
    Saved plot -> compare_ES1_ES2.png

[ES1_GS]  overlap x-range: 1.640e+16 to 4.688e+17
    max |diff|    = 4.9212e+26
    mean |diff|   = 3.7313e+25
    max rel diff  = 90.65 %
    mean rel diff = 0.85 %
    Saved 